# Partitioning & Shuffle Mechanics

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

No connection boilerplate is needed — `spark.master`, `spark.sql.shuffle.partitions`, and `spark.sql.adaptive.enabled` are already baked into this cluster's `spark-defaults.conf` from what you set in the control panel.

In [ ]:
from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("partitioning-shuffle").getOrCreate()
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.sql.adaptive.enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

## 1. Build a keyed dataset spread across input partitions

We deliberately spread rows across more input partitions than there are workers, so the initial (pre-shuffle) stage's tasks fan out across the whole cluster.

In [ ]:
NUM_ROWS = 2_000_000
NUM_KEYS = 500
NUM_INPUT_PARTITIONS = 24  # > worker_count so tasks distribute across all workers

rdd = spark.sparkContext.parallelize(range(NUM_ROWS), NUM_INPUT_PARTITIONS).map(
    lambda i: Row(key=f"key-{i % NUM_KEYS}", value=float(i % 997))
)
df = spark.createDataFrame(rdd)
print("Input partitions:", df.rdd.getNumPartitions())
df.show(5)

## 2. Form a hypothesis before running

Before executing the next cell, jot down (in this markdown cell or on paper) your answers:

- Will `groupBy("key").agg(...)` require a shuffle? Why?
- How many post-shuffle partitions do you expect, given `spark.sql.shuffle.partitions`?
- Do you expect `shuffleReadBytes` to roughly equal `shuffleWriteBytes`?

_(Write your answers here before running the cell below.)_

In [ ]:
result = df.groupBy("key").agg(
    F.count("*").alias("cnt"),
    F.avg("value").alias("avg_value"),
)

total = result.count()  # materialize the job
print(f"groupBy().agg() produced {total} distinct keys.")

## 3. Read the physical plan

Look for the `Exchange` node — that's the shuffle boundary. Compare it against your hypothesis.

In [ ]:
result.explain(mode="formatted")

## 4. Verify against ground truth via the Spark REST API

Check this against your hypothesis, then go look at `http://localhost:4040` (Stages + SQL tabs) and `http://localhost:8080` (worker count) directly.

In [ ]:
import json
import urllib.request

app_id = spark.sparkContext.applicationId
url = f"http://localhost:4040/api/v1/applications/{app_id}/stages"
with urllib.request.urlopen(url, timeout=5) as resp:
    stages = json.loads(resp.read().decode("utf-8"))

for stage in stages:
    read_bytes = stage.get("shuffleReadBytes", 0)
    write_bytes = stage.get("shuffleWriteBytes", 0)
    if read_bytes or write_bytes:
        print(
            f"stage {stage.get('stageId')} (attempt {stage.get('attemptId')}): "
            f"shuffleReadBytes={read_bytes} shuffleWriteBytes={write_bytes} "
            f"numTasks={stage.get('numTasks')}"
        )

## 5. Optional: change `spark.sql.shuffle.partitions` and compare

Try setting a much smaller or larger value and re-running the aggregation, then compare the post-shuffle partition count and task duration distribution in the Spark UI.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

result2 = df.groupBy("key").agg(F.count("*").alias("cnt"))
result2.count()
result2.explain(mode="formatted")

# Reset back to the cluster default before continuing.
spark.conf.set("spark.sql.shuffle.partitions", 200)